# Week 3 on Colab — three ways to fine-tune a real model

The first time in the course you train a model you did not build. Everything that does the training is code you wrote: the loop from week 1, and this week's `tunelab/lora.py`.

**Runtime → Change runtime type → T4 GPU**, then run the cells top to bottom. About twenty minutes.

Do not start until `pytest week-03` is green on your laptop and pushed.

## 1. Get your code

In [ ]:
import os

REPO_URL = "..."   # the address of YOUR repo

if not os.path.exists("course") and os.path.basename(os.getcwd()) != "course":
    !git clone {REPO_URL} course
if os.path.basename(os.getcwd()) != "course":
    %cd course
!pip install -q -e . pytest peft

## 2. Drive and GPU

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
os.makedirs("/content/drive/MyDrive/tunelab", exist_ok=True)
ADAPTER = "/content/drive/MyDrive/tunelab/week-03-adapter.pt"

import torch

assert torch.cuda.is_available(), "No GPU. Runtime -> Change runtime type -> T4 GPU, then start again from the top."
print(torch.cuda.get_device_name(0))

## 3. Is your LoRA the library's LoRA?

The same parity tests you ran on your laptop, now against whatever version of `peft` Colab has today. They should still pass. If they do not, tell your instructor before going on: the library may have changed a convention.

In [ ]:
!pytest week-03/milestone/test_parity.py -q

## 4. Open the real model

The first run downloads about 270 MB.

In [ ]:
import importlib.util
from pathlib import Path

from tunelab.pretrained import layer_table, load_pretrained
from tunelab.text import make_get_batch

MODEL = "HuggingFaceTB/SmolLM2-135M-Instruct"
REVISION = "12fd25f77366fa6b3b4b768ec3050bf629380bac"


def fresh():
    """The pretrained model, exactly as downloaded, on the GPU."""
    model, tokenizer = load_pretrained(MODEL, REVISION)
    return model.to("cuda"), tokenizer


model, tokenizer = fresh()
for name, shape, count in layer_table(model)[:8]:
    print(f"{count:>12,}  {str(shape):<16} {name}")
print(f"{sum(count for _, _, count in layer_table(model)):>12,}  total")

## 5. Two kinds of text

**New:** Shakespeare, which you want the model to get better at. **Old:** plain modern English, which it is already good at and which you will never train on. The old text is the thermometer for forgetting.

In [ ]:
def encode(path):
    return torch.tensor(tokenizer(Path(path).read_text()).input_ids, device="cuda")


shakespeare = make_get_batch(encode("data/tiny.txt"), block_size=128, batch_size=16)
general = make_get_batch(encode("data/general.txt"), block_size=128, batch_size=16, val_frac=0.5)

x, y = shakespeare("train")
print(x.shape, x.device)
print(repr(tokenizer.decode(x[0][:40])))

## 6. Does your code work on a GPU?

Your tests ran on a CPU. This puts adapters into the real model on the GPU and runs one batch through it. If it fails with **`Expected all tensors to be on the same device`**, your `LoRALinear` is creating `A` and `B` on the CPU: Day 3's README, "three practical details".

In [ ]:
from tunelab.lora import inject_lora, lora_state_dict, merge_lora

probe, _ = fresh()
print("layers wrapped:", inject_lora(probe, ("q_proj", "v_proj"), rank=8, alpha=16))
logits, loss = probe(x, y)
assert logits.is_cuda
print("works on the GPU. Loss on Shakespeare before any training:", round(loss.item(), 3))
del probe

## 7. The experiment

The same model, the same 300 steps of Shakespeare, three ways. Before you run it, **write your prediction in `logs/run-log.md`**: which method learns Shakespeare best, which forgets the most plain English, and roughly how much GPU memory each will need.

A fresh copy of the model is loaded for each method, so none of them starts from another's result.

In [ ]:
spec = importlib.util.spec_from_file_location("compare", "week-03/milestone/compare.py")
compare = importlib.util.module_from_spec(spec)
spec.loader.exec_module(compare)

STEPS = 300
LEARNING_RATE = {"full": 5e-5, "last_block": 5e-5, "lora": 2e-4}
results = {}


def experiment(method, rank=8):
    model, _ = fresh()
    torch.cuda.reset_peak_memory_stats()
    result = compare.run(
        method, model, shakespeare, general, steps=STEPS, lr=LEARNING_RATE[method],
        rank=rank, alpha=2 * rank, targets=("q_proj", "v_proj"), last_block="layers.29.",
    )
    result["peak_gpu_mb"] = torch.cuda.max_memory_allocated() / 1e6
    return result, model


for method in ("full", "last_block", "lora"):
    results[method], tuned = experiment(method)
    r = results[method]
    print(f'{method:<11} trainable {r["trainable"]:>11,}  shakespeare {r["new_before"]:.3f} -> {r["new_after"]:.3f}  '
          f'general {r["old_before"]:.3f} -> {r["old_after"]:.3f}  peak GPU {r["peak_gpu_mb"]:,.0f} MB')

lora_model = tuned      # the last one trained: LoRA at rank 8. Kept for the cells below.

## 8. What does rank buy?

In [ ]:
for rank in (1, 4, 16):
    results[f"lora r={rank}"], _ = experiment("lora", rank=rank)

print(f'{"":<12}{"trainable":>12}{"% of model":>12}{"shakespeare":>14}{"general":>10}{"peak GPU MB":>13}')
for name, r in results.items():
    print(f'{name:<12}{r["trainable"]:>12,}{100 * r["trainable"] / r["total"]:>11.3f}%'
          f'{r["new_after"]:>14.3f}{r["old_after"]:>10.3f}{r["peak_gpu_mb"]:>13,.0f}')
print(f'\nbefore any training: shakespeare {results["full"]["new_before"]:.3f}, general {results["full"]["old_before"]:.3f}')

## 9. The adapter is a small file

In [ ]:
torch.save(lora_state_dict(lora_model), ADAPTER)
adapter_mb = os.path.getsize(ADAPTER) / 1e6
model_mb = sum(count for _, _, count in layer_table(model)) * 4 / 1e6
print(f"adapter: {adapter_mb:.2f} MB    model: {model_mb:,.0f} MB    ratio: 1 to {model_mb / adapter_mb:,.0f}")

## 10. Merge it

After merging there should be no adapters left, and the losses should not have moved.

In [ ]:
from tunelab.loop import estimate_loss

before = estimate_loss(lora_model, lambda split: (x, y), "val", batches=1)
print("layers merged:", merge_lora(lora_model))
after = estimate_loss(lora_model, lambda split: (x, y), "val", batches=1)
print(f"same batch, before merging {before:.4f}, after {after:.4f}")
assert abs(before - after) < 1e-3

## 11. Hear the difference

The same opening, continued by the model as downloaded and by your tuned one.

In [ ]:
def continue_text(m, prompt, n=80):
    ids = tokenizer(prompt, return_tensors="pt").input_ids.to("cuda")
    out = m.hf.generate(ids, max_new_tokens=n, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0])


prompt = "ROMEO:\n"
print("--- as downloaded ---")
print(continue_text(model, prompt))
print("\n--- tuned ---")
print(continue_text(lora_model, prompt))

## 12. Write it up

Copy the table from cell 8, the adapter size from cell 9 and both samples into `week-03/milestone/WRITEUP.md`, and answer its three questions. Add a row per method to `logs/run-log.md`. Commit from your laptop.